In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_openai import  ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
from tavily import  TavilyClient

tavily_client= TavilyClient()

results=tavily_client.search("Sivaprasad Valluru")
results

In [ ]:
from langchain.tools import tool

@tool
def websearch(query):
    """Perform a web search using Tavily and return the results."""
    results=tavily_client.search(query)
    return results

@tool
def add(a, b):
    """Add two numbers."""
    return a + b

websearch

In [ ]:
llm_withtools=llm.bind_tools([websearch,add])

result= llm_withtools.invoke("Who won t20 world cup 2026? and also add 10 and 20")
result

In [ ]:
result.tool_calls

In [ ]:
from langchain.agents import  create_agent
from langchain_tavily import  TavilySearch

agent = create_agent(
    model=llm,
    tools=[TavilySearch(),add]
)

agent

In [ ]:
result= agent.invoke(
    {
        "messages":[
            {"role": "user", "content": "Tell me who won t20 world cup in 2026. Also add 10 and 20."}
        ]
    }
)

result

In [ ]:
result['messages']

In [ ]:
result['messages'][1].tool_calls

In [ ]:
result['messages'][-1].content

In [ ]:
from langchain_community.utilities.openweathermap import OpenWeatherMapAPIWrapper

wrapper = OpenWeatherMapAPIWrapper()
wrapper.run("mumbai")

In [ ]:
from pydantic import BaseModel,Field

class WeatherInput(BaseModel):
    x: str = Field(..., description="The city to get the weather for.")


@tool(args_schema=WeatherInput)
def get_weather(x):
    """Get the current weather."""
    return wrapper.run(x)

In [ ]:
agent= create_agent(
    model=llm,
    tools=[websearch,add,get_weather]
)

response=agent.invoke(
    {
        "messages":[
            {"role": "user", "content": " what is temperature of mumbai today? if temperature is less than 30 degree celsius then add 10 and 20."}
        ]
    }
)
response

In [ ]:
response['messages'][1].tool_calls

In [ ]:
from langchain_community.agent_toolkits import  SQLDatabaseToolkit
from langchain_community.utilities import SQLDatabase

db= SQLDatabase.from_uri("sqlite:///fraud_demo.sqlite3")

toolkit= SQLDatabaseToolkit(db=db, llm=llm)
tools=toolkit.get_tools()
tools


In [ ]:
agent= create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
SQL RULES (apply to every query):
- Use only the provided SQL tools.
- ALWAYS list tables, then inspect schema for tables you touch.
- Add LIMIT when listing rows unless the question says otherwise.
- NEVER run INSERT, UPDATE, DELETE, DROP, or other DML.

ROLE:
Return-frequency risk analyst on an e-commerce fraud team.

GOAL:
When given a shopper email, count their non-rejected returns in the last 30 days
and assign a frequency risk_score (0–100) using the rubric below.

BACKSTORY:
You investigate whether a customer is returning items too often — a common sign
of policy abuse. You query a read-only SQLite database with tables: users, orders,
order_items, returns. You focus ONLY on return counts and timing. Ignore refund
dollar amounts and return-reason text; other specialists on the team handle those.

WORKFLOW:
1. Query the returns table — count rows where:
   - user_email matches the shopper
   - return_date is within the last 30 days
   - status is NOT 'REJECTED'
2. Map return_count to risk_score
   (pick one integer inside the band; use the higher end at the top of the range):
   - 0 returns       → 0–10   (low)
   - 1–2 returns     → 10–25  (normal)
   - 3–5 returns     → 25–50  (monitor)
   - 6–9 returns     → 50–75  (elevated)
   - 10+ returns     → 75–100 (high — likely abuse)

OUTPUT FORMAT:
1. Write 2–4 sentences in plain language summarizing what you found.
2. End with ONE line of strict JSON only (no markdown fences). Example shape:
   {"check_type": "frequency", "risk_score": 0, "reason": "short explanation",
    "details": {"return_count": 0, "period_days": 30}}

JSON field rules:
- risk_score: integer 0–100 from the rubric (use the real count, not a placeholder)
- reason: one sentence citing the count and band
  (e.g. "12 returns in 30 days — high frequency")
- details.return_count: exact integer from your SQL query
- details.period_days: always 30



    """
)

In [ ]:
result=agent.invoke(
    {
        "messages":[
            {"role": "user","content": (
            f"Return ticket — frequency risk check\n"
            f"Shopper email: fraudster@example.com \n\n"
            "How many non-rejected returns has this shopper made in the last 30 days? "
            "Give a brief explanation and a frequency risk score."
        ),}
        ]
    }   
)

In [ ]:
result

In [ ]:
Markdown(result['messages'][-1].content)

In [ ]:
result= agent.invoke(
    {
        "messages":[
            {"role": "user", "content": "Display the portfolio of John Doe."}
        ]
    }
)
result

In [ ]:
result['messages'][6].tool_calls

In [ ]:
from IPython.display import display, Markdown


display(Markdown(result['messages'][-1].content))

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

conn = sqlite3.connect("checkpoint.sqlite" , check_same_thread=False)

sqlite_saver = SqliteSaver(conn=conn)
sqlite_saver.setup()


agent = create_agent(
    model=llm,
    checkpointer=sqlite_saver
)

In [ ]:
config = {"configurable": {"thread_id": "1"}}

result = agent.invoke(
    {
        "messages":[
            {"role": "user", "content": "My Name is Siva"}
        ]
    }, config=config
)
result

In [ ]:
result = agent.invoke(
    {
        "messages":[
            {"role": "user", "content": "What is my name"}
        ]
    }, config=config
)
result

In [ ]:
USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson", 
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
        "support_tier": "Priority"
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard", 
        "balance": 1200,
        "email": "bob@example.com",
        "support_tier": "Standard"
    }
}

In [ ]:
from dataclasses import dataclass

@dataclass
class UserContext:
    user_id: str

In [ ]:
from langchain.tools import ToolRuntime

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:

    """ 
    Retrieves account information for the user associated with the current context.

    Returns:
        str: A formatted string containing the user's account information.
    """

    ctx = runtime.context

    if ctx.user_id in USER_DATABASE:
        user_info = USER_DATABASE[ctx.user_id]
        return (
            f"Account Information for {user_info['name']}:\n"
            f"- Account Type: {user_info['account_type']}\n"
            f"- Balance: ${user_info['balance']}\n"
            f"- Email: {user_info['email']}\n"
            f"- Support Tier: {user_info['support_tier']}"
        )

    

In [ ]:
agent = create_agent(
    model=llm,
    tools=[get_account_info],
    context_schema= UserContext
)

In [ ]:
result = agent.invoke(
    {
        "messages":[
            {"role": "user", "content": "What is my account balance?"}
        ]
    },
    context=UserContext(user_id="user123")
)

result

In [ ]:
result['messages'][1].tool_calls

In [ ]:

@tool
def save_to_ltm(data: str, runtime: ToolRuntime[UserContext]) -> str:
    """ 
    Saves data to the long-term memory (LTM) for the user associated with the current context.

    use this tool when user shares personal information, explicit approvals  and user preferences
  
    """

    store= runtime.store

    user_id = runtime.context.user_id

    ns= ("app","memory",user_id)

    item= store.get(ns, "ltm")
    if item is None:
        messages= []
    else:
        messages= item.value

    messages.append(data)

    store.put(ns, "ltm", messages)

    return f"Data saved to long-term memory for user {user_id}."


@tool
def retrieve_from_ltm(runtime: ToolRuntime[UserContext]) -> str:
    """ 
    Retrieves data from the long-term memory (LTM) for the user associated with the current context.

   
    """

    store= runtime.store

    user_id = runtime.context.user_id

    ns= ("app","memory",user_id)

    item= store.get(ns, "ltm")
    if item is None:
        return f"No data found in long-term memory for user {user_id}."
    else:
        messages= item.value
        return "\n".join(messages)

In [33]:
from langgraph.store.memory import  InMemoryStore
from langgraph.checkpoint.memory import  InMemorySaver

agent = create_agent(
    model=llm,
    tools=[save_to_ltm, retrieve_from_ltm],
    context_schema= UserContext,
    checkpointer=InMemorySaver(),
    store= InMemoryStore(),

    system_prompt="""

You are a helpful assistant that can answer any queries.
If the user shares personal information, preferences, or explicit approvals,
save that information using the save_to_ltm tool.
Before answering a query when prior facts might matter, retrieve long-term
memory using retrieve_from_ltm and use that context in your answer.

    """
)



In [39]:
config = {"configurable": {"thread_id": "22"}}

result= agent.invoke(
    {
        "messages":[
            {"role": "user", "content": "I love to eat  biriyani when i go to hyderabad"}
        ]
    }, config=config
    ,context=UserContext(user_id="user123")
)

result

{'messages': [HumanMessage(content='I love to eat  biriyani when i go to hyderabad', additional_kwargs={}, response_metadata={}, id='9c61dee0-cd8c-4285-89b2-fe5ecaa67e99'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 183, 'total_tokens': 210, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5b1a5ed202', 'id': 'chatcmpl-E41C1VM2DQKR3vEogGpuXEvcr9b0M', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f8408-e169-7e80-af93-fd0c285c06f0-0', tool_calls=[{'name': 'save_to_ltm', 'args': {'data': 'User loves to eat biriyani when they go to Hyderabad.'}, 'id': 'call_Jvg9f2PkUsKdGwKWfwMP

In [36]:
result['messages'][1].tool_calls

[{'name': 'save_to_ltm',
  'args': {'data': 'User loves to eat vada pav when they come to Mumbai.'},
  'id': 'call_yfKnPR6NgOXK05UxzaFekHPg',
  'type': 'tool_call'}]

In [40]:
config = {"configurable": {"thread_id": "333"}}

result= agent.invoke(
    {
        "messages":[
            {"role": "user", "content": "What do I prefer to eat in Hyderabad?"}
        ]
    }, config=config
    ,context=UserContext(user_id="user123")
)

result

{'messages': [HumanMessage(content='What do I prefer to eat in Hyderabad?', additional_kwargs={}, response_metadata={}, id='cbf27489-d773-4269-939c-8024f14c1024'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 179, 'total_tokens': 191, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5b1a5ed202', 'id': 'chatcmpl-E41CL8uY0z30IDYia5bc5MKzUlrBs', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f8409-32ec-7453-9517-00cfee323fc1-0', tool_calls=[{'name': 'retrieve_from_ltm', 'args': {}, 'id': 'call_nemFYdTNdpY4wfnIqn2LVic7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat

In [38]:
result['messages'][1].tool_calls

[{'name': 'retrieve_from_ltm',
  'args': {},
  'id': 'call_S2YKWrdVVgm9utpxfnxWIzri',
  'type': 'tool_call'}]